# Ark+ (Swin-L) baseline on CheXpert — Colab

**Before running:**
1. `Runtime > Change runtime type` → GPU. T4 works fine for the current config (see below); for the real 768px Ark+ baseline later, use the fastest available (H100 > A100 > L4 > T4) — needs Colab Pro/Pro+ or pay-as-you-go compute units.
2. `USE_ARK_WEIGHTS` (cell 12) is currently **`False`** — training Swin-L from random init, no checkpoint needed. Set it to `True` once you have the real Ark+ checkpoint (`Ark6_swinLarge768_ep50.pth.tar`, requested via the Google Form in the [Ark repo README](https://github.com/jlianglab/Ark)) — cell 12 checks Drive first, then falls back to an upload dialog if not found there yet.

Cell 14 has a `NUM_CLASSES` toggle: **14** (current default) trains the model head on all 14 CheXpert labels but only reports AUROC/etc. on the 5 competition classes — the original Ark+ paper's protocol. **5** trains the head on just the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) directly; section 2's patch 3 makes that possible. Built on the official [jlianglab/Ark](https://github.com/jlianglab/Ark) finetuning code.

**On speed:** the repo's training loop originally created a `NativeScaler` but never actually wrapped the forward pass in `torch.autocast` — no mixed-precision benefit despite appearances. Section 2's patch 6 fixes this for real (`bfloat16` autocast — native on Ampere/A100 and even stronger on Hopper/H100, no gradient scaling needed unlike fp16). With `USE_ARK_WEIGHTS=False`, cell 14 auto-switches to a cheap 224px config (`BATCH_SIZE=32`); flipping to `True` switches to the heavy 768px config Ark+ actually needs (`BATCH_SIZE=4`, conservative but proven to work on A100 — tune down further if you hit an OOM).

**On accuracy:** cell 14 also has three toggles (`USE_PER_CLASS_UNCERTAIN_LABEL`, `USE_CLASS_WEIGHTED_LOSS`, `USE_COOCCURRENCE_LOSS`, all `True` by default) for data-driven improvements built from gaps observed in the initial baseline run plus recent chest X-ray literature — see section 2 (patches 10-12) and section 4 for details. Set any to `False` to A/B against the plain baseline.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Kaggle credentials + dataset download

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Clone Ark+ repo, patch it, and install dependencies
The original repo pins 2021-era package versions. `timm==0.5.4` and `albumentations==0.4.3` are kept exact — the repo's README warns newer timm versions changed the Swin Transformer architecture enough to break pretrained-weight loading, and the dataloader uses transform names albumentations removed later. We patch one `np.bool` alias (removed in numpy>=1.24) instead of downgrading numpy globally, to avoid conflicting with Colab's preinstalled stack.

Patches applied here (all verified locally first):
- `models.py` imports `UperNet_swin` from a module that isn't actually included in this repo folder — a dead, unused import that crashes on load. Removed.
- `dataloader.py`'s `CheXpert` dataset always reads all 14 label columns. We add an optional `class_indices` argument so it can select just the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) in the right order — this is what powers the `NUM_CLASSES=5` option in cell 12's toggle. With `NUM_CLASSES=14` (the current default), `class_indices` is left `None` and the dataset falls back to its original all-14-columns behavior.
- Three data-driven improvements (patches 10-12), motivated by gaps observed in the initial baseline run and recent chest X-ray literature — each independently toggleable in cell 14:
  - **Per-pathology uncertain-label strategy** (patch 10): `dataloader.py`'s `uncertain_label` can now be a list (one strategy per class) instead of only a single global string — the original CheXpert paper's own U-Ones/U-Zeros ablation found the best strategy is pathology-dependent.
  - **Class-weighted loss** (patches 11/12): a `pos_weight` on the BCE loss, computed from the real training-set class balance, to counter the imbalance behind low-AP classes like Consolidation.
  - **Co-occurrence-aware regularization** (patches 11/12): a small penalty (`utils.CoOccurrenceBCELoss`) that nudges the classifier head's weight vectors for commonly co-occurring pathologies to point in more similar directions — a simplified, batch-size-independent take on hierarchical/dependency-aware losses in recent literature (e.g. HBCE, arXiv:2502.03591).

This cell is safe to re-run (e.g. after a kernel restart) — the clone and patches are skipped/no-op if already present.

In [ ]:
import os, sys

# Install the same dependencies the training code needs (this doesn't change: copying our own
# code below doesn't install third-party packages).
!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a np.sctypes
# reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import misc` import
# (removed in modern scipy) in the same file. Both crash on *import*, so we can't `import imgaug`
# first to locate the file -- that import is exactly what crashes. Using importlib.util.find_spec
# instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# The actual CheXpert training code (dataloader/trainer/engine/models) is pulled straight from
# Drive instead of re-deriving ~15 sed/python patches against a live jlianglab/Ark clone every
# run. This copy already has everything: base compatibility fixes, all 6 models wired in, and
# the full frozen protocol (320px support for every architecture, patient-level held-out test
# split, stratified ablation subset, ECE metric, configurable AMP dtype, per-trial seeding) --
# kept in sync with the canonical copy at github.com/fei7fei/chexpert-baselines. Every notebook
# reads from this exact same folder, so the underlying code is guaranteed byte-identical across
# all 6 models -- required for the frozen-protocol comparison to be valid.
from google.colab import drive
drive.mount('/content/drive')

_DRIVE_CODE_SRC = '/content/drive/MyDrive/Project/models/Repos/ark_plus_repo/Ark_Plus/Finetuning'
if not os.path.exists('/content/chexpert_code'):
    import shutil
    shutil.copytree(_DRIVE_CODE_SRC, '/content/chexpert_code')
%cd /content/chexpert_code
print('Loaded pre-patched CheXpert training code (frozen protocol v1) from Drive.')

In [ ]:
import torch, timm
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
print('timm', timm.__version__)
from timm.models.swin_transformer import SwinTransformer
m = SwinTransformer(num_classes=5, img_size=768, patch_size=4, window_size=12, embed_dim=192,
                     depths=(2,2,18,2), num_heads=(6,12,24,48))
print('Swin-L (768, window12) built OK, params:', sum(p.numel() for p in m.parameters()))
del m

## 3. Mount Google Drive for checkpoint persistence
`/content` is ephemeral — if the runtime disconnects or gets recycled (e.g. Colab Compute Units running out), everything in it is gone, checkpoints included. This mounts Drive and points training's save location there instead, so progress survives even a full runtime loss. Re-running cell 14 after a restart will then automatically resume from the last saved epoch (via `args.resume`) instead of starting over. Mounted first (before the checkpoint step below) so that step can check Drive for an already-uploaded Ark+ checkpoint instead of always requiring the upload dialog.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/ark_plus_chexpert_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints and results will be saved to:', DRIVE_CKPT_DIR)

## 3b. Ark+ pretrained checkpoint
Set `USE_ARK_WEIGHTS = True` for the real Ark+ baseline (requires `Ark6_swinLarge768_ep50.pth.tar`, requested via the Google Form in the [Ark repo README](https://github.com/jlianglab/Ark)). Leave `False` to train Swin-L from random init instead.

Checks Drive first (`DRIVE_CKPT_DIR/Ark6_swinLarge768_ep50.pth.tar`) before falling back to the upload dialog — upload once, and it's there for every future session without re-uploading. Also more robust than relying on the upload dialog alone: `files.upload()` is the same class of browser-dependent Colab widget (`google.colab.userdata`) that silently failed earlier this session when connected through the VS Code↔Colab bridge rather than the browser.

In [ ]:
import os, shutil

USE_ARK_WEIGHTS = False  # set True once you have the real Ark+ checkpoint (see cell 11)

CKPT_FILENAME = 'Ark6_swinLarge768_ep50.pth.tar'
DRIVE_CKPT_PATH = os.path.join(DRIVE_CKPT_DIR, CKPT_FILENAME)
CKPT_PATH = f'/content/Ark/Ark_Plus/Finetuning/{CKPT_FILENAME}'

if not USE_ARK_WEIGHTS:
    CKPT_PATH = None
    print('No checkpoint — will train Swin-L from random init.')
elif os.path.exists(DRIVE_CKPT_PATH):
    shutil.copy(DRIVE_CKPT_PATH, CKPT_PATH)
    print('Found checkpoint on Drive, using it directly:', CKPT_PATH)
else:
    print('Checkpoint not found on Drive -- falling back to upload dialog.')
    print('(If this hangs or fails silently through the VS Code<->Colab bridge, try running')
    print('this same cell from browser Colab instead, then rerun it here once uploaded.)')
    from google.colab import files
    uploaded = files.upload()  # select Ark6_swinLarge768_ep50.pth.tar
    ckpt_name = list(uploaded.keys())[0]
    os.rename(ckpt_name, CKPT_PATH)
    shutil.copy(CKPT_PATH, DRIVE_CKPT_PATH)  # so future sessions skip the upload entirely
    print('Uploaded and cached to Drive for next time:', DRIVE_CKPT_PATH)
    print('Using checkpoint:', CKPT_PATH)

## 4. Run finetuning on the CheXpert benchmark
Trains Swin-L per the `NUM_CLASSES` toggle set in the cell below, reporting AUROC/MCC/AP/F1 on the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) each epoch either way. Adjust `BATCH_SIZE`/`EPOCHS`/`PATIENCE` there for your GPU and time budget.

Three toggles just below `NUM_CLASSES` control the data-driven improvements from section 2 (patches 10-12) — all default `True`, set any to `False` to A/B against the plain baseline:
- `USE_PER_CLASS_UNCERTAIN_LABEL` — per-pathology uncertain-label strategy instead of one global `"LSR-Ones"`.
- `USE_CLASS_WEIGHTED_LOSS` — `pos_weight` on the BCE loss, computed from the actual training-set class balance.
- `USE_COOCCURRENCE_LOSS` — co-occurrence-aware regularization on the classifier head's weights.

`args.exp_name` (and so the checkpoint directory) encodes which of these are active, so flipping a toggle starts a fresh run instead of accidentally resuming into a mismatched one.

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

# ===== Frozen protocol v1 -- identical across all 6 models, fix this before running anything =====
INPUT_SIZE = 320                    # native CheXpert-small short side -- avoids gratuitous up/downsampling
UNCERTAIN_LABEL = "Zeros"           # U-Zeros for main results; flip to "Ones" for the appendix sensitivity check
EPOCHS = 50
SEEDS_SCREENING = [13]              # 1 seed: cheap ranking pass on the 20% ablation subset
SEEDS_HEADLINE = [13, 42, 2024]     # 3 seeds: confirmation pass on the full training pool

args.model_name = "swin_large"
args.init = "random"
args.pretrained_weights = None
args.input_size = INPUT_SIZE
args.img_size = INPUT_SIZE
args.num_class = 14   # train the head on all 14 CheXpert labels; report only on the 5 competition
                       # classes via test_diseases below (original Ark+ protocol convention)
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.key = "teacher"
args.batch_size = 32   # "largest that fits" per protocol -- reduce if you hit OOM at 320px,
                                  # or wire in gradient checkpointing (not done here)
args.epochs = EPOCHS
args.patience = EPOCHS           # no early stopping -- every model trains the full 50 epochs so run
                                  # lengths stay comparable across the whole comparison
args.opt = "adamw"
args.lr = 5e-5
args.weight_decay = 0.05
args.sched = "cosine"
args.warmup_epochs = 1
args.cooldown_epochs = 0
args.uncertain_label = UNCERTAIN_LABEL
args.test_every_epoch = True
args.save_every_epoch_checkpoint = True
args.amp_dtype = "fp16"
args.device = "cuda"
args.resume = True
args.workers = 2

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
test_diseases_5 = [diseases_14.index(c) for c in competition_5]

from dataloader import CheXpert, build_transform_classification, build_protocol_splits, filter_csv_by_patients
from engine import classification_engine

# Patient-level held-out test (carved from train) + stratified 20% ablation subset -- computed
# once and cached to a *shared* Drive folder (not this model's own checkpoint folder) so every
# one of the 6 models' notebooks reads the exact same patient partition, in whatever order they
# happen to run. That's what makes the cross-model comparison valid rather than 6 separate
# anecdotes on 6 different splits.
PROTOCOL_SHARED_DIR = '/content/drive/MyDrive/chexpert_protocol_v1'
os.makedirs(PROTOCOL_SHARED_DIR, exist_ok=True)

train_csv = os.path.join(args.data_dir, "train.csv")
splits = build_protocol_splits(train_csv, competition_5, cache_path=os.path.join(PROTOCOL_SHARED_DIR, "protocol_split.csv"))

def _cached_filter(name, patient_set):
    p = os.path.join(PROTOCOL_SHARED_DIR, f"{name}.csv")
    if not os.path.exists(p):
        filter_csv_by_patients(train_csv, patient_set, p)
    return p

train_pool_csv = _cached_filter("train_pool", splits["train_pool"])
held_out_test_csv = _cached_filter("held_out_test", splits["held_out_test"])
ablation_subset_csv = _cached_filter("ablation_subset", splits["ablation_subset"])
official_val_csv = os.path.join(args.data_dir, "valid.csv")

args.train_list = train_pool_csv
args.val_list = official_val_csv     # the official 234-study set -- used for per-epoch threshold
                                      # tuning/checkpoint selection only, never for final reporting
args.test_list = held_out_test_csv   # final reporting -- held out from train, never seen in training

def make_ds(mode, file_path):
    return CheXpert(images_path=args.data_dir, file_path=file_path,
        augment=build_transform_classification(normalize=args.normalization, mode=mode, crop_size=args.input_size, resize=args.img_size),
        uncertain_label=args.uncertain_label, unknown_label=args.unknown_label)

dataset_val = make_ds("valid", args.val_list)
dataset_test = make_ds("test", args.test_list)
print(f"val (official, threshold tuning): {len(dataset_val)}  |  test (held-out, final report): {len(dataset_test)}")

model_path = os.path.join(DRIVE_CKPT_DIR, "Models/Classification", args.data_set)
output_path = os.path.join(DRIVE_CKPT_DIR, "Outputs/Classification", args.data_set)

# ===== Tier 1: screening -- cheap, single seed, ranks variants on the 20% subset =====
print("\n=== TIER 1: screening (1 seed, 20% stratified subset) ===")
args.exp_name = "_protocol1_arkplus_screening"
args.num_trial = len(SEEDS_SCREENING)
args.start_index = 0
args.seed_list = SEEDS_SCREENING
dataset_train_screen = make_ds("train", ablation_subset_csv)
print(f"screening train size: {len(dataset_train_screen)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_screen, dataset_val, dataset_test, test_diseases=test_diseases_5)

# ===== Tier 2: headline -- 3 seeds, full training data. Report both tiers; the two-tier split =====
# ===== is itself part of what makes ~20 runs comparable rather than 20 anecdotes. =====
print("\n=== TIER 2: headline (3 seeds, full training pool) ===")
args.exp_name = "_protocol1_arkplus_headline"
args.num_trial = len(SEEDS_HEADLINE)
args.start_index = 0
args.seed_list = SEEDS_HEADLINE
dataset_train_full = make_ds("train", train_pool_csv)
print(f"headline train size: {len(dataset_train_full)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_full, dataset_val, dataset_test, test_diseases=test_diseases_5)

## 5. Results
Per-epoch AUROC prints during training above. Final summary (mean/std AUC, MCC, AP, F1 across the 5 classes) lands in `Outputs/Classification/CheXpert/*_results.txt`; checkpoints in `Models/Classification/CheXpert/` — both now inside `DRIVE_CKPT_DIR` on your Google Drive (see cell 12), so they survive a runtime loss. If cell 14 gets cut off mid-run, just re-run it — `args.resume=True` picks back up from the last saved epoch automatically.

The next cell shows the clean per-epoch table (Epoch, train_loss, val_loss, F1, AUC, recall, accuracy) as a DataFrame — patch 9 (cell 7) writes this to its own CSV alongside the verbose text log, rewritten after every epoch so it's readable even from a run that got interrupted partway through.

In [ ]:
!find "{DRIVE_CKPT_DIR}/Outputs" -type f
!echo ---
!cat "{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert"/*_results.txt

In [ ]:
import glob
import pandas as pd

_csvs = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv"))
if not _csvs:
    print("No epoch table found yet -- run cell 14 through at least one epoch first.")
else:
    _latest = _csvs[-1]
    print("Showing:", _latest)
    display(pd.read_csv(_latest))

## 6. Publication-style comparison table
Builds a `Method | Acc | Sen | F1 | AUC` table (the format common in chest X-ray classification papers) from every `*_results.txt` file found in `Outputs/Classification/CheXpert/` — one row per experiment (baseline, baseline+improvements, or any ablation you run by toggling cell 14's `USE_*` flags). "Sen" (Sensitivity) is the same quantity as `Recall` in the per-epoch table above; renamed here to match the convention in most published tables.

Note: with `args.num_trial=1` (the current default), there's only a single run per configuration, so there's no real cross-run standard deviation to report — the `(mean ± std)` style you see in papers comes from averaging several independently-seeded trials. Bump `args.num_trial` in cell 14 (e.g. to 3) and rerun if you want genuine ± error bars later; that costs proportionally more compute.

In [ ]:
import re, glob, os
import pandas as pd

_pattern = re.compile(
    r"aAUC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aMCC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAP = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aF1 = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aRecall = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAccuracy = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aECE = (-?[\d.]+)\((-?[\d.]+)\)"
)

def _tier_label(fname):
    # exp_name convention (see the config+training cell): _protocol1_<model>_screening /
    # _protocol1_<model>_headline. Two-tier discipline: screening is a cheap single-seed
    # ranking pass on the 20% stratified subset; headline is the confirmed 3-seed result on
    # the full training pool. Both are reported explicitly and separately below -- an
    # unlabeled/blended number is exactly what an unexplained-subset result looks like.
    base = os.path.basename(fname).replace("_results.txt", "")
    if base.endswith("_screening"):
        return "Screening (1 seed, 20% subset)"
    if base.endswith("_headline"):
        return "Headline (3 seeds, 100% data)"
    return base.lstrip("_").replace("_", " ")  # fallback for any other exp_name found here

rows = []
for f in sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert/*_results.txt")):
    text = open(f).read()
    m = _pattern.search(text)
    if not m:
        print("Could not parse (no 'Mean / STD over All trials' line found -- run may still be in progress):", f)
        continue
    auc, auc_std, mcc, mcc_std, ap, ap_std, f1, f1_std, recall, recall_std, acc, acc_std, ece, ece_std = map(float, m.groups())
    rows.append({
        "Tier": _tier_label(f),
        "Acc (%)": f"{acc*100:.1f} +/- {acc_std*100:.1f}",
        "Sen (%)": f"{recall*100:.1f} +/- {recall_std*100:.1f}",
        "F1 (%)": f"{f1*100:.1f} +/- {f1_std*100:.1f}",
        "AUC (%)": f"{auc*100:.1f} +/- {auc_std*100:.1f}",
        "ECE (%)": f"{ece*100:.1f} +/- {ece_std*100:.1f}",
    })

if not rows:
    print("No results files found yet -- run the config+training cell through completion first.")
else:
    df = pd.DataFrame(rows).set_index("Tier")
    print("Two-tier protocol results, reported separately as required: screening (single seed,")
    print("cheap ranking pass on the 20% subset) and headline (3-seed confirmation on the full")
    print("training pool). Screening's +/- is trivially 0.0 (n=1) -- only headline's spread is")
    print("meaningful; shown for both so the table format stays consistent.\n")
    display(df)